In [ ]:
# Imports
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
# Config
DATA_PATH = Path('projects/Medication Non Adherence/Medication_Non_Adherence.csv')
RANDOM_STATE = 42
TEST_SIZE = 0.2

# LIME settings (keep modest for notebook runtime; increase if desired)
LIME_NUM_EXPLAINED = 20
LIME_NUM_SAMPLES = 500
LIME_NUM_FEATURES = 10

np.random.seed(RANDOM_STATE)

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)

# Drop accidental index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

display(df.head())
print('Rows:', len(df))
print('Columns:', df.shape[1])
print('ADH distribution:', df['ADH'].value_counts(dropna=False).to_dict())

In [ ]:
# Build target: non-adherence = 1 if ADH == -1
y = (df['ADH'] == -1).astype(int)

# Feature selection: drop ID/name-like and date fields, plus the label
DROP_COLS = {'ADH', 'Id', 'FIRST', 'LAST', 'BIRTHDATE', 'DEATHDATE', 'Life Span'}
present_drop_cols = [c for c in df.columns if c in DROP_COLS]
X = df.drop(columns=present_drop_cols)

# Drop constant columns
nunique = X.nunique(dropna=False)
const_cols = nunique[nunique <= 1].index.tolist()
if const_cols:
    X = X.drop(columns=const_cols)

print('Features used:', X.shape[1])
display(X.head())

In [ ]:
# Train/test split (stratified)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
(train_idx, test_idx), = splitter.split(X, y)

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print('Train size:', len(X_train), 'Test size:', len(X_test))
print('Train class balance:', y_train.value_counts().to_dict())

In [ ]:
# Preprocessing: numeric median imputation, categorical most-frequent + one-hot
categorical_cols = [c for c in X_train.columns if X_train[c].dtype == 'object']
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

numeric_pipe = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, numeric_cols),
        ('cat', categorical_pipe, categorical_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

In [ ]:
# Fit models
pos = int(y_train.sum())
neg = int((y_train == 0).sum())
scale_pos_weight = float(neg / max(pos, 1))

xgb = XGBClassifier(
    n_estimators=250,
    learning_rate=0.08,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

lgbm = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced',
    verbosity=-1,
    verbose=-1,
)

pipe_xgb = Pipeline(steps=[('preprocess', preprocess), ('model', xgb)])
pipe_lgbm = Pipeline(steps=[('preprocess', preprocess), ('model', lgbm)])

pipe_xgb.fit(X_train, y_train)
pipe_lgbm.fit(X_train, y_train)

models = {'xgboost': pipe_xgb, 'lightgbm': pipe_lgbm}

In [ ]:
# Evaluate
def evaluate(pipe, X_eval, y_eval):
    prob = pipe.predict_proba(X_eval)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y_eval, prob)),
        'avg_precision': float(average_precision_score(y_eval, prob)),
        'accuracy': float(accuracy_score(y_eval, pred)),
        'f1': float(f1_score(y_eval, pred)),
        'classification_report': classification_report(y_eval, pred, digits=4),
    }

metrics_by_model = {name: evaluate(pipe, X_test, y_test) for name, pipe in models.items()}
metrics_by_model

In [ ]:
# Pick best model by ROC AUC
best_name = max(metrics_by_model.items(), key=lambda kv: kv[1]['roc_auc'])[0]
best_pipe = models[best_name]
print('Best model:', best_name)

## LIME interpretability

We explain the model using LIME in the **preprocessed feature space** (after one-hot encoding).
Then we aggregate absolute contributions across explanations to produce a global list of key predictors.

In [ ]:
# Prepare preprocessed matrices and feature names
preprocessor = best_pipe.named_steps['preprocess']
model = best_pipe.named_steps['model']

X_train_p = preprocessor.transform(X_train)
X_test_p = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out().tolist()
feature_names[:10], len(feature_names)

In [ ]:
# Create LIME explainer
explainer = LimeTabularExplainer(
    training_data=X_train_p,
    feature_names=feature_names,
    class_names=['adherent', 'nonadherent'],
    mode='classification',
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

rng = np.random.default_rng(RANDOM_STATE)
n_explain = min(LIME_NUM_EXPLAINED, X_test_p.shape[0])
idx = rng.choice(X_test_p.shape[0], size=n_explain, replace=False)

feature_score = {}
feature_count = {}
examples = []

def predict_fn(arr: np.ndarray) -> np.ndarray:
    return model.predict_proba(arr)

print(f'Running LIME on {n_explain} rows (samples={LIME_NUM_SAMPLES}, features={LIME_NUM_FEATURES})')

for n_done, i in enumerate(idx, start=1):
    exp = explainer.explain_instance(
        X_test_p[i],
        predict_fn,
        num_features=LIME_NUM_FEATURES,
        num_samples=LIME_NUM_SAMPLES,
    )
    pairs = exp.as_list(label=1)
    examples.append({
        'row_index': int(i),
        'y_true': int(y_test.iloc[i]),
        'pred_proba_nonadherent': float(predict_fn(X_test_p[i].reshape(1, -1))[0, 1]),
        'top_features': pairs,
    })

    for name, weight in pairs:
        feature_score[name] = feature_score.get(name, 0.0) + float(abs(weight))
        feature_count[name] = feature_count.get(name, 0) + 1

    if n_done % 10 == 0 or n_done == n_explain:
        print(f'LIME progress: {n_done}/{n_explain} explanations')

ranked = sorted(feature_score.items(), key=lambda kv: kv[1], reverse=True)
key_predictors = [
    {'feature': name, 'score': float(score), 'occurrences_in_explanations': int(feature_count.get(name, 0))}
    for name, score in ranked[:LIME_NUM_FEATURES]
]

key_predictors

In [ ]:
# Summary requested: records analyzed + number of key predictors
print('Patient records analyzed:', len(df))
print('Key predictors identified (LIME):', len(key_predictors))

for i, item in enumerate(key_predictors, start=1):
    print(f"{i:02d}. {item['feature']} (occurrences={item['occurrences_in_explanations']}, score={item['score']:.4f})")

In [ ]:
# Optional: write a compact JSON summary (saved in the project folder)
summary = {
    'dataset': {
        'patient_records': int(len(df)),
        'target_definition': 'nonadherent = 1 if ADH == -1 else 0',
        'class_balance': {
            'nonadherent': int(y.sum()),
            'adherent': int((y == 0).sum()),
        },
    },
    'models': {
        'best_model': best_name,
        'metrics': metrics_by_model,
    },
    'lime': {
        'lime_num_explained': int(n_explain),
        'lime_num_samples': int(LIME_NUM_SAMPLES),
        'lime_num_features': int(LIME_NUM_FEATURES),
        'key_predictors': key_predictors,
        'example_explanations': examples[:10],
    },
}

out_path = Path('projects/Medication Non Adherence/run_summary.json')
out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Wrote:', out_path)